## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import log_loss

## Reading Data

In [ ]:
# reading CSVs
vehicle_df = pd.read_csv('../data/localdata/Traffic_Crashes_Vehicles.csv')
people_df = pd.read_csv('../data/localdata/Traffic_Crashes_People.csv')
crash_df = pd.read_csv('../data/localdata/Traffic_Crashes_Crashes.csv')

## Data Cleaning

In [ ]:
# Crashes DataFrame

crash_df.drop(columns=['WORK_ZONE_I', 'WORK_ZONE_TYPE', 'DOORING_I', 'WORKERS_PRESENT_I',
                                          'PHOTOS_TAKEN_I', 'STATEMENTS_TAKEN_I', 'RD_NO', 'REPORT_TYPE', 'NOT_RIGHT_OF_WAY_I', 'CRASH_DATE_EST_I'], axis=1, inplace=True)
crash_df.dropna(subset = ['INJURIES_TOTAL', 'LATITUDE', 'MOST_SEVERE_INJURY', 
                          'STREET_DIRECTION', 'BEAT_OF_OCCURRENCE'], axis=0, inplace=True)
crash_df.columns = crash_df.columns.str.title()
crash_df.columns = crash_df.columns.str.replace('_', ' ')

In [ ]:
# Vehicles DataFrame

vehicle_df = vehicle_df[['CRASH_UNIT_ID', 'CRASH_RECORD_ID', 'CRASH_DATE', 'UNIT_NO', 'UNIT_TYPE',
                       'VEHICLE_YEAR', 'VEHICLE_USE', 'VEHICLE_TYPE', 'MANEUVER', 'OCCUPANT_CNT',
                         'AREA_00_I', 'AREA_01_I', 'AREA_02_I', 'AREA_03_I', 'AREA_04_I',
                       'AREA_05_I', 'AREA_06_I', 'AREA_07_I', 'AREA_08_I', 'AREA_09_I', 'AREA_10_I', 'AREA_11_I',
                       'AREA_12_I', 'AREA_99_I', 'FIRST_CONTACT_POINT']]
vehicle_df.dropna(subset = ['VEHICLE_USE', 'FIRST_CONTACT_POINT', 'UNIT_TYPE'], axis=0, inplace=True)
vehicle_df['VEHICLE_YEAR'].fillna(value = 'Unknown', inplace=True)

vehicle_df.columns = vehicle_df.columns.str.title()
vehicle_df.columns = vehicle_df.columns.str.replace('_', ' ')

In [ ]:
# People DataFrame

people_df.drop(columns=['RD_NO', 'CELL_PHONE_USE', 'PEDPEDAL_ACTION', 
                        'PEDPEDAL_VISIBILITY', 'PEDPEDAL_LOCATION', 'SEAT_NO', 
                        'HOSPITAL', 'EMS_AGENCY', 'EMS_RUN_NO', 'BAC_RESULT', 'BAC_RESULT VALUE', 
                        'DRIVERS_LICENSE_STATE', 'DRIVERS_LICENSE_CLASS', 'CITY', 'STATE', 'ZIPCODE']
               , axis=1, inplace=True)

people_df.dropna(subset = ['AIRBAG_DEPLOYED', 'EJECTION', 'INJURY_CLASSIFICATION', 'VEHICLE_ID', 
                           'SAFETY_EQUIPMENT', 'SEX'], axis=0, inplace=True)

people_df.columns = people_df.columns.str.title()
people_df.columns = people_df.columns.str.replace('_', ' ')

## Feature Engineering Crash Score

In [ ]:
# Injury Score

crash_df[['Injuries Total', 'Injuries Fatal', 'Injuries Incapacitating', 
            'Injuries Non Incapacitating', 'Injuries Reported Not Evident']][crash_df['Injuries Total']!=0][:50]

crash_df['total injured'] = crash_df['Injuries Fatal'] + crash_df['Injuries Incapacitating'] + crash_df['Injuries Non Incapacitating'] + crash_df['Injuries Reported Not Evident']

# Checking if added up columns in 'total injured' make up 'Injuries Total'
(crash_df['total injured'] == crash_df['Injuries Total']).value_counts()

# Injury Score column is a linear combination of the factors that make up Injuries Total simply multiplied by a constant depending on their severity
crash_df['Injury Score'] = crash_df['Injuries Fatal']*7 + crash_df['Injuries Incapacitating']*3 + crash_df['Injuries Non Incapacitating']*2 + crash_df['Injuries Reported Not Evident']

# Accounting Material Damage of Car and Surrounding Areas )
# Crash Score 

crash_df['Damage'].value_counts()
mapping = {'OVER $1,500': 3, '$501 - $1,500': 2, "$500 OR LESS": 1}
crash_df['Damage_ODE'] = crash_df['Damage'].map(mapping)
crash_df['Injury Score'] = crash_df['Injury Score'] + crash_df['Damage_ODE']*3
crash_df['Crash Score'] = crash_df['Injury Score']

In [ ]:
print(crash_df.shape)
print(vehicle_df.shape)

## Merging Vehicle and Crash DataFrames

In [ ]:
vehicles_crashes_df = vehicle_df.merge(crash_df, on = 'Crash Record Id', how = 'inner')
vehicles_crashes_df.drop_duplicates(subset='Crash Record Id', inplace=True)

In [ ]:
categorical = ['Weather Condition', 'Roadway Surface Cond', 'Road Defect', 'Alignment',
                       'Traffic Control Device', 'Device Condition', 'Crash Hour', 'Trafficway Type', 'Maneuver'] #potentiall vehicle defects

ohe = OneHotEncoder(drop='first')
ohe.fit(vehicles_crashes_df[categorical])
categorical_encoded = pd.DataFrame(ohe.transform(vehicles_crashes_df[categorical]).todense(),
                               columns=ohe.get_feature_names())
# for Nick's version of OneHotEncoder, since its newer, the attribute is get_feature_names_out but get_feature_name for bobby and mike's version

In [ ]:
area_df = vehicles_crashes_df[['Crash Record Id','Area 00 I', 'Area 01 I', 'Area 02 I', 'Area 03 I', 'Area 04 I',
                       'Area 05 I', 'Area 06 I', 'Area 07 I', 'Area 08 I', 'Area 09 I', 'Area 10 I', 'Area 11 I',
                       'Area 12 I', 'Area 99 I', 'Latitude', 'Longitude']]

In [ ]:
vehicles_crashes_df.to_csv('../data/localdata/vehicles_crashes.csv')

In [ ]:
area_df.head()

In [ ]:
area_df.shape

In [ ]:
multi_yes = area_df.columns[area_df.eq('Y').sum() > 1]

In [ ]:
mask = area_df[multi_yes].eq('Y').sum(axis=1) <= 1

In [ ]:
filtered_area_df = area_df[mask]

In [ ]:
filtered_area_df.shape

In [ ]:
filtered_area_df.head()

In [ ]:
for column in filtered_area_df.columns[1:15]:
    mapping = {'Y': 1, 'N': np.nan}
    
    area_column = filtered_area_df[column].map(mapping)
    
    filtered_area_df[column] = area_column
print(filtered_area_df)

In [ ]:
filtered_area_df.fillna(0, inplace=True)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
encoder = OrdinalEncoder()


In [ ]:
filtered_area_df.isna().sum()

In [ ]:
def fill_area(row):
    if row["Area 00 I"] == 1:
        return 0
    elif row["Area 01 I"] == 1:
        return 1
    elif row["Area 02 I"] == 1:
        return 2
    elif row["Area 03 I"] == 1:
        return 3
    elif row["Area 04 I"] == 1:
        return 4
    elif row["Area 05 I"] == 1:
        return 5
    elif row["Area 06 I"] == 1:
        return 6
    elif row["Area 07 I"] == 1:
        return 7
    elif row["Area 08 I"] == 1:
        return 8
    elif row["Area 09 I"] == 1:
        return 9
    elif row["Area 10 I"] == 1:
        return 10
    elif row["Area 11 I"] == 1:
        return 11
    elif row["Area 12 I"] == 1:
        return 12
    elif row["Area 99 I"] == 1:
        return 99
    else:
        pass

filtered_area_df['Area Number'] = filtered_area_df.apply(fill_area, axis=1)

filtered_area_df['Area Number']

In [ ]:
filtered_area_df['Area Number'].value_counts()

In [ ]:
filtered_area_df.head()

In [ ]:
filtered_area_df.to_csv('../data/localdata/filtered_area.csv')